# Machine Learning & Model Development — Online Retail II
**Axlero Group 3 | Model Development workstream**  
**Churn target:** No purchase during the 60 days after the observation date.

## Purpose
This notebook documents a reproducible model-development workflow for the team's Online Retail II churn-prediction project. It focuses on feature preparation, chronological data splitting, model training, validation-based selection, and saving a reusable model artifact.

### Target and feature windows
- Historical feature lookback: 90 days.
- Prediction horizon: 60 days.
- `churn_risk = 1`: customer makes no purchase in the 60 days after the observation date.
- `churn_risk = 0`: customer makes at least one purchase in that future 60-day period.

This is an inactivity-risk proxy, not proof of permanent customer cancellation.

### Input
Upload `online_retail_II_cleaned.csv` to the notebook runtime. Required columns:
`Invoice`, `InvoiceDate`, `CustomerID`, `StockCode`, `Quantity`, `Price`.
`Revenue` is optional and will be calculated when absent.

### Reproducibility
Run cells from top to bottom. The latest 20% of snapshot dates are held out for final evaluation and are not used for model selection.

## 1. Imports and configuration

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score, confusion_matrix,
    classification_report
)

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
DATA_FILE = "online_retail_II_cleaned.csv"
LOOKBACK_DAYS = 90
PREDICTION_DAYS = 60

if not os.path.exists(DATA_FILE):
    raise FileNotFoundError(
        f"{DATA_FILE} not found. Upload the cleaned CSV to the Colab session first."
    )

raw = pd.read_csv(DATA_FILE)
print("Raw cleaned-data shape:", raw.shape)
print("Columns:", raw.columns.tolist())

## 2. Validate and prepare transactions

In [ ]:
required = {"Invoice", "InvoiceDate", "CustomerID", "StockCode", "Quantity", "Price"}
missing = required - set(raw.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

tx = raw.copy()
tx["InvoiceDate"] = pd.to_datetime(tx["InvoiceDate"], errors="coerce")
for col in ["CustomerID", "Quantity", "Price"]:
    tx[col] = pd.to_numeric(tx[col], errors="coerce")
if "Revenue" not in tx.columns:
    tx["Revenue"] = tx["Quantity"] * tx["Price"]
tx["Revenue"] = pd.to_numeric(tx["Revenue"], errors="coerce")

tx = tx.dropna(subset=["InvoiceDate", "CustomerID", "Invoice", "StockCode", "Quantity", "Price", "Revenue"])
tx = tx[
    (tx["CustomerID"] > 0) & (tx["Quantity"] > 0) &
    (tx["Price"] > 0) & (tx["Revenue"] > 0)
].copy()
tx["CustomerID"] = tx["CustomerID"].astype("int64")
tx["Invoice"] = tx["Invoice"].astype(str)
tx = tx.sort_values("InvoiceDate").reset_index(drop=True)

print("Transactions used:", len(tx))
print("Unique customers:", tx["CustomerID"].nunique())
print("Date range:", tx["InvoiceDate"].min(), "to", tx["InvoiceDate"].max())

## 3. Build customer snapshots and 60-day target

In [ ]:
first_date = tx["InvoiceDate"].min().normalize()
last_date = tx["InvoiceDate"].max().normalize()
snapshot_dates = pd.date_range(
    first_date + pd.offsets.MonthEnd(0),
    last_date - pd.Timedelta(days=PREDICTION_DAYS),
    freq="ME"
)
if len(snapshot_dates) < 5:
    raise ValueError(
        "Fewer than five monthly snapshots are available. "
        "Check the input date range; a chronological train/validation/test split needs enough dates."
    )

snapshots = []
for date in snapshot_dates:
    hist_start = date - pd.Timedelta(days=LOOKBACK_DAYS)
    future_end = date + pd.Timedelta(days=PREDICTION_DAYS)
    hist = tx[(tx["InvoiceDate"] > hist_start) & (tx["InvoiceDate"] <= date)]
    future = tx[(tx["InvoiceDate"] > date) & (tx["InvoiceDate"] <= future_end)]
    if hist.empty:
        continue

    future_buyers = set(future["CustomerID"].unique())
    grouped = hist.groupby("CustomerID")
    frame = grouped.agg(
        recency_days=("InvoiceDate", lambda x: (date - x.max()).days),
        order_frequency=("Invoice", "nunique"),
        total_quantity=("Quantity", "sum"),
        monetary_value=("Revenue", "sum"),
        unique_products=("StockCode", "nunique"),
        transaction_lines=("Invoice", "size"),
        first_purchase=("InvoiceDate", "min"),
        last_purchase=("InvoiceDate", "max")
    ).reset_index()

    invoice_counts = hist.groupby("CustomerID")["Invoice"].nunique()
    frame["average_order_value"] = frame["CustomerID"].map(
        frame.set_index("CustomerID")["monetary_value"] / invoice_counts
    )
    frame["customer_tenure_days"] = (frame["last_purchase"] - frame["first_purchase"]).dt.days
    frame["observation_date"] = date
    frame["churn_risk"] = (~frame["CustomerID"].isin(future_buyers)).astype(int)
    snapshots.append(frame)

data = pd.concat(snapshots, ignore_index=True).replace([np.inf, -np.inf], np.nan)
print("Snapshot rows:", len(data))
print("Snapshot dates:", data["observation_date"].nunique())
print("60-day inactivity target rate:", round(data["churn_risk"].mean(), 4))
display(data.head())

## 4. Feature set and chronological split

In [ ]:
FEATURES = [
    "recency_days", "order_frequency", "total_quantity", "monetary_value",
    "unique_products", "transaction_lines", "average_order_value",
    "customer_tenure_days"
]
dates = np.array(sorted(data["observation_date"].unique()))
n = len(dates)
train_end = max(1, int(n * 0.60))
valid_end = max(train_end + 1, int(n * 0.80))
valid_end = min(valid_end, n - 1)
if train_end >= valid_end or valid_end >= n:
    raise ValueError("Not enough distinct snapshot dates for a 60/20/20 chronological split.")

train_dates = dates[:train_end]
valid_dates = dates[train_end:valid_end]
test_dates = dates[valid_end:]

train = data[data.observation_date.isin(train_dates)].copy()
valid = data[data.observation_date.isin(valid_dates)].copy()
test = data[data.observation_date.isin(test_dates)].copy()

X_train, y_train = train[FEATURES], train["churn_risk"].astype(int)
X_valid, y_valid = valid[FEATURES], valid["churn_risk"].astype(int)
X_test, y_test = test[FEATURES], test["churn_risk"].astype(int)

for label, target in [("train", y_train), ("validation", y_valid), ("test", y_test)]:
    print(label, "rows:", len(target), "positive rate:", round(target.mean(), 4))
    if target.nunique() < 2:
        raise ValueError(f"{label} split has only one target class. Review date coverage.")
print("Train dates:", train_dates[0], "to", train_dates[-1])
print("Validation dates:", valid_dates[0], "to", valid_dates[-1])
print("Test dates:", test_dates[0], "to", test_dates[-1])

## 5. Train candidate machine-learning models

In [ ]:
candidates = {
    "Dummy baseline": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE))
    ]),
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=300, min_samples_leaf=3,
            class_weight="balanced_subsample", random_state=RANDOM_STATE, n_jobs=-1
        ))
    ]),
    "Extra Trees": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", ExtraTreesClassifier(
            n_estimators=300, min_samples_leaf=3,
            class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
        ))
    ])
}

fitted = {}
rows = []
for name, estimator in candidates.items():
    estimator.fit(X_train, y_train)
    fitted[name] = estimator
    pred = estimator.predict(X_valid)
    prob = estimator.predict_proba(X_valid)[:, 1]
    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_valid, pred),
        "Balanced_Accuracy": balanced_accuracy_score(y_valid, pred),
        "Precision": precision_score(y_valid, pred, zero_division=0),
        "Recall": recall_score(y_valid, pred, zero_division=0),
        "F1": f1_score(y_valid, pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_valid, prob),
        "PR_AUC": average_precision_score(y_valid, prob)
    })

validation_results = pd.DataFrame(rows).sort_values("F1", ascending=False)
display(validation_results.round(4))
validation_results.to_csv("ml_validation_model_comparison.csv", index=False)
best_name = validation_results.iloc[0]["Model"]
print("Selected model (validation F1):", best_name)

## 6. Random Forest hyperparameter experiment (validation only)

In [ ]:
# A small, explicit experiment. Each candidate is fitted on training data
# and compared on validation F1. The final test set remains untouched.
rf_candidates = [
    {"n_estimators": 200, "max_depth": None, "min_samples_leaf": 2},
    {"n_estimators": 300, "max_depth": 12, "min_samples_leaf": 3},
    {"n_estimators": 400, "max_depth": 20, "min_samples_leaf": 5},
]
tuning_rows = []
for params in rf_candidates:
    estimator = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            **params, class_weight="balanced_subsample",
            random_state=RANDOM_STATE, n_jobs=-1
        ))
    ])
    estimator.fit(X_train, y_train)
    pred = estimator.predict(X_valid)
    prob = estimator.predict_proba(X_valid)[:, 1]
    tuning_rows.append({
        **params,
        "Validation_F1": f1_score(y_valid, pred, zero_division=0),
        "Validation_ROC_AUC": roc_auc_score(y_valid, prob)
    })

tuning_results = pd.DataFrame(tuning_rows).sort_values("Validation_F1", ascending=False)
display(tuning_results.round(4))
tuning_results.to_csv("random_forest_tuning_results.csv", index=False)

## 7. Refit selected model and evaluate on the held-out test period

In [ ]:
# Use the best validation-selected model; do not select by test metrics.
selected = fitted[best_name]
X_train_valid = pd.concat([X_train, X_valid], axis=0)
y_train_valid = pd.concat([y_train, y_valid], axis=0)
selected.fit(X_train_valid, y_train_valid)

test_pred = selected.predict(X_test)
test_prob = selected.predict_proba(X_test)[:, 1]
test_metrics = pd.DataFrame([{
    "Selected_Model": best_name,
    "Accuracy": accuracy_score(y_test, test_pred),
    "Balanced_Accuracy": balanced_accuracy_score(y_test, test_pred),
    "Precision": precision_score(y_test, test_pred, zero_division=0),
    "Recall": recall_score(y_test, test_pred, zero_division=0),
    "F1": f1_score(y_test, test_pred, zero_division=0),
    "ROC_AUC": roc_auc_score(y_test, test_prob),
    "PR_AUC": average_precision_score(y_test, test_prob)
}])
display(test_metrics.round(4))
test_metrics.to_csv("ml_final_test_metrics.csv", index=False)
print(classification_report(y_test, test_pred, zero_division=0))

## 8. Confusion matrix and feature importance

In [ ]:
cm = confusion_matrix(y_test, test_pred)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm)
ax.set_title("Final test confusion matrix")
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_xticks([0, 1], ["Purchase", "No purchase"])
ax.set_yticks([0, 1], ["Purchase", "No purchase"])
for (i, j), value in np.ndenumerate(cm):
    ax.text(j, i, str(value), ha="center", va="center")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig("ml_confusion_matrix.png", dpi=160, bbox_inches="tight")
plt.show()

if best_name in ["Random Forest", "Extra Trees"]:
    fitted_estimator = selected.named_steps["model"]
    importance_values = fitted_estimator.feature_importances_
    importance = pd.DataFrame({"Feature": FEATURES, "Importance": importance_values}).sort_values("Importance", ascending=False)
    display(importance)
    importance.to_csv("ml_feature_importance.csv", index=False)
    importance.sort_values("Importance").plot(
        x="Feature", y="Importance", kind="barh", legend=False,
        figsize=(8, 5), title=f"{best_name} feature importance"
    )
    plt.tight_layout()
    plt.savefig("ml_feature_importance.png", dpi=160, bbox_inches="tight")
    plt.show()
else:
    print("Feature importance output is only generated here for tree-based selected models.")

## 9. Save model artifact and run metadata

In [ ]:
artifact = {
    "model": selected,
    "selected_model_name": best_name,
    "feature_columns": FEATURES,
    "lookback_days": LOOKBACK_DAYS,
    "prediction_days": PREDICTION_DAYS,
    "target_definition": "No purchase during the 60 days following the observation date",
    "selection_metric": "Validation F1",
    "training_validation_dates": [str(pd.Timestamp(train_dates[0]).date()), str(pd.Timestamp(valid_dates[-1]).date())],
    "test_dates": [str(pd.Timestamp(test_dates[0]).date()), str(pd.Timestamp(test_dates[-1]).date())]
}
joblib.dump(artifact, "ml_churn_model_artifact.joblib")

summary = [
    "MACHINE LEARNING MODEL DEVELOPMENT SUMMARY",
    "Target: no purchase in the 60 days after each observation date",
    f"Feature lookback: {LOOKBACK_DAYS} days",
    f"Selected model: {best_name}",
    "Selection basis: validation F1",
    f"Final test F1: {test_metrics.iloc[0]['F1']:.4f}",
    f"Final test ROC-AUC: {test_metrics.iloc[0]['ROC_AUC']:.4f}",
    "Test period was held out from model selection."
]
with open("ml_model_development_summary.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(summary))
print("\n".join(summary))
print("Saved model artifact and summary.")

## 10. Model development conclusions and limitations

Complete the written interpretation using the actual validation and test outputs.

- Compare the validation scores of all candidate models.
- Explain why validation F1 was selected as the model-selection metric.
- Discuss final held-out test performance without using it to re-select the model.
- Interpret feature importance only as model-specific contribution, not causation.
- The 60-day no-purchase target is an inactivity-risk proxy, not a verified permanent churn event.
- Transaction history may not include all customer interactions or purchases outside the dataset.
- The latest 60 days of transactions cannot be labelled because the full future outcome period is unavailable.

### Expected generated files
- `ml_validation_model_comparison.csv`
- `random_forest_tuning_results.csv`
- `ml_final_test_metrics.csv`
- `ml_confusion_matrix.png`
- `ml_feature_importance.csv` (if selected model is tree-based)
- `ml_feature_importance.png` (if selected model is tree-based)
- `ml_churn_model_artifact.joblib`
- `ml_model_development_summary.txt`